# arXiv Preprints Analysis

How do economics and software-engineering preprints differ?

Compare output and author counts by year, apply a title-based rule for machine learning, and examine publication indicators. Publication status inferred from available metadata is a proxy.


In [ ]:
import pandas as pd

df = pd.read_csv("data/arxiv_econ_se.csv.gz")

df.head()

In [ ]:
df["field"] = "cs.SE"
df.loc[df["categories"].str.startswith("econ"), "field"] = "econ"
import numpy as np
df["field"] = np.where(df["categories"].str.startswith("econ"), "econ", "cs.SE")

In [ ]:
df["field"] = df["categories"].str.split().str[0].str.replace(r"^econ\..*", "econ", regex=True)
df["year"] = pd.to_datetime(df["submitted"]).dt.year

df.groupby(["field", "year"]).size()

In [ ]:
check_field = (df["field"] == np.where(df["categories"].str.startswith("econ"), "econ", "cs.SE")).all()

check_year = (df["year"] == pd.to_datetime(df["submitted"]).dt.year).all()

print("Field:", check_field)
print("Year:", check_year)
print("All correct:", check_field and check_year)

In [ ]:
preprints_by_year = df.groupby(["field", "year"]).size()
sum(preprints_by_year)

In [ ]:
print(preprints_by_year)
print("Total preprints:", preprints_by_year.sum())

In [ ]:
sum(preprints_by_year) == len(df)

In [ ]:
authors_with_brackets = df[df["authors"].str.contains(r"\(", na=False)]
print(authors_with_brackets["authors"])

In [ ]:
import re

def clean_authors(text):
    if pd.isna(text):
        return text

    text = str(text)

    while re.search(r"\([^()]*\)", text):
        text = re.sub(r"\([^()]*\)", "", text)

    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+,", ",", text)
    text = re.sub(r",\s*", ", ", text)

    return text.strip()

df["authors_clean"] = df["authors"].apply(clean_authors)

print("Original parentheses:", df["authors"].str.contains(r"\(", na=False).sum())
print("Remaining parentheses:", df["authors_clean"].str.contains(r"\(", na=False).sum())

df.loc[df["authors"] != df["authors_clean"], ["authors", "authors_clean"]].head(10)

In [ ]:
df["author_list"] = df["authors_clean"].str.replace(" and ", ",", regex=False).str.split(",")

authors_long = df.explode("author_list").copy()
authors_long["author_list"] = authors_long["author_list"].str.strip()
authors_long = authors_long[authors_long["author_list"].notna() & (authors_long["author_list"] != "")]

people_by_year = authors_long.groupby(["year", "field"])["author_list"].nunique().unstack()

n_people_econ = int(authors_long[authors_long["field"] == "econ"]["author_list"].nunique())
n_people_se = int(authors_long[authors_long["field"] == "cs.SE"]["author_list"].nunique())


people_by_year

In [ ]:
is_ml_first = (df["title"] + " " + df["abstract"]).str.contains("machine learning", case=False, regex=False)

ml_share_first = df.assign(is_ml=is_ml_first).groupby(["field", "year"])["is_ml"].mean()

print(is_ml_first)
print(ml_share_first)

In [ ]:
print(len(is_ml_first) == len(df))
print(is_ml_first.sum())

In [ ]:
ml_pattern = (
    "machine learning|"
    "learning to rank|"
    "supervised learning by classification|"
    "supervised learning by regression|"
    "structured outputs|"
    "cost-sensitive learning|"
    "cluster analysis|"
    "anomaly detection|"
    "mixture modell?ing|"
    "topic modell?ing|"
    "source separation|"
    "motif discovery|"
    "dimensionality reduction and manifold learning|"
    "sequential decision making|"
    "inverse reinforcement learning|"
    "apprenticeship learning|"
    "multi-agent reinforcement learning|"
    "adversarial learning"
)

df["is_ml"] = (df["title"] + " " + df["abstract"]).str.contains(ml_pattern, case=False, regex=True, na=False)

ml_share = df.groupby(["field", "year"])["is_ml"].mean()


print("ML papers:", df["is_ml"].sum())
print(ml_share)

In [ ]:
papers = df[(df["field"] == "cs.SE") & (df["year"] == 2022)]
ml_papers = papers[papers["is_ml"] == True].head(20)
non_ml_papers = papers[papers["is_ml"] == False].head(20)

print("ML papers:")
print(ml_papers[["title", "abstract"]].to_string())

print("\nNon-ML papers:")
print(non_ml_papers[["title", "abstract"]].to_string())

In [ ]:
n_wrong_matches = 1
n_missed = 6

In [ ]:
print(ml_papers.loc[ml_papers["title"] == "Digital Twin: From Concept to Practice", ["title", "abstract"]].to_string(index=False))

print(non_ml_papers.loc[non_ml_papers["title"] == "Toward the Analysis of Graph Neural Networks", ["title", "abstract"]].to_string(index=False))

In [ ]:
counts = df.groupby(["field", "year"]).agg(
    total=("is_ml", "size"),
    ml=("is_ml", "sum")
)

for field in ["cs.SE", "econ"]:
    total_growth = counts.loc[(field, 2024), "total"] - counts.loc[(field, 2018), "total"]
    ml_growth = counts.loc[(field, 2024), "ml"] - counts.loc[(field, 2018), "ml"]
    print(field, round(ml_growth / total_growth * 100, 2), "%")

In [ ]:
for column in ["journal_ref", "doi", "comments"]:
    print("\n", column)
    print(df[column].value_counts().head(15))

In [ ]:
df["published"] = "no evidence"

df.loc[df["doi"].notna(), "published"] = "doi"
df.loc[df["journal_ref"].notna(), "published"] = "journal_ref"

published_share = df.groupby(["year", "field"])["published"].apply(
    lambda x: (x != "no evidence").mean()
)

In [ ]:
comment_evidence = df["comments"].str.contains(
    "accepted|published|forthcoming|to appear",
    case=False,
    na=False
)

print("Publication evidence in comments:", comment_evidence.sum())

print("New evidence not counted before:",
      ((df["published"] == "no evidence") & comment_evidence).sum())

In [ ]:
df.loc[
    (df["published"] == "no evidence") & comment_evidence,
    ["title", "comments"]
].sample(20, random_state=42)

In [ ]:
published_share_with_comments = (
    (df["published"] != "no evidence") | comment_evidence
).groupby([df["year"], df["field"]]).mean()

print("Original share:")
print(published_share)

print("Share with comments:")
print(published_share_with_comments)

print("Difference:")
print((published_share_with_comments - published_share) * 100)

In [ ]:
import os
os.makedirs("results", exist_ok=True)

people_by_year.to_csv("results/authors.csv")
ml_share.to_csv("results/ml.csv")
published_share_with_comments.to_csv("results/publications.csv")